In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('unsw-nb15/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('unsw-nb15/UNSW_NB15_testing-set.csv')

# 移除无关列
train_df.drop(columns=['id', 'label'], inplace=True)
test_df.drop(columns=['id', 'label'], inplace=True)

# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())
print("Test missing values:", test_df.isnull().sum().sum())

# One-hot 编码列
cols = ['proto', 'state', 'service']

def one_hot(df, cols):
    for col in cols:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df.drop(col, axis=1, inplace=True)
    return df

# 合并数据进行统一处理
combined_data = pd.concat([test_df, train_df], ignore_index=True)

# 分离目标变量
target = combined_data.pop('attack_cat')

# One-hot 编码
combined_data = one_hot(combined_data, cols)

# Min-Max 归一化
def normalize(df):
    result = df.copy()
    for col in df.columns:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > min_val:
            result[col] = (df[col] - min_val) / (max_val - min_val)
    return result

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data['Class'] = target

# 检查是否有缺失值
assert not normalized_data.isnull().values.any(), "归一化后的数据存在缺失值"

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 将目标变量转换为整数编码
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

Train missing values: 0
Test missing values: 0
(257673, 196)


In [2]:
import torch
import torch.nn as nn

class ParallelDilatedTCN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, 16, 32, padding='same', dilation=1),
                nn.BatchNorm1d(16),
                nn.GELU()
            ),
            nn.Sequential(
                nn.Conv1d(in_channels, 16, 64, padding='same', dilation=3),
                nn.BatchNorm1d(16),
                nn.GELU()
            ),
            nn.Sequential(
                nn.Conv1d(in_channels, 16, 96, padding='same', dilation=9),
                nn.BatchNorm1d(16),
                nn.GELU()
            )
        ])
        self.fusion_gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(48, 48, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        branch_outs = [branch(x) for branch in self.branches]
        merged = torch.cat(branch_outs, dim=1)
        return merged * self.fusion_gate(merged)



import torch
import torch.nn as nn
import torch.nn.functional as F

# ========================= Bi-ResAtt-LSTM ========================= #
class BiResAttLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.2):
        super(BiResAttLSTM, self).__init__()

        # 双向 LSTM
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # 自适应注意力门控
        self.att_gate = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),  # [B, L, H*2] -> [B, L, H]
            nn.GELU(),
            nn.Linear(hidden_size, 1),               # [B, L, H] -> [B, L, 1]
            nn.Sigmoid()
        )

        # 残差连接
        self.res_proj = nn.Linear(input_size, hidden_size * 2)  # 输入和输出维度对齐

    def forward(self, x):
        # x: [B, L, input_size]
        res = self.res_proj(x)                  # 残差映射
        lstm_out, _ = self.bilstm(x)            # 双向 LSTM 输出 [B, L, H*2]

        att_weights = self.att_gate(lstm_out)   # 注意力权重 [B, L, 1]
        attended = lstm_out * att_weights       # 加权 LSTM 输出 [B, L, H*2]

        return attended + res                   # 残差连接增强 [B, L, H*2]

class NSLKDDModel(nn.Module):
    def __init__(self, input_dim=196, num_classes=10, hidden_size=32):  # 新增hidden_size参数
        super().__init__()
        
        # 改进的TCN模块（保持原样）
        self.tcn = nn.Sequential(
            ParallelDilatedTCN(in_channels=1),
            nn.AdaptiveMaxPool1d(input_dim//4),  # 输出长度=input_dim//4
            nn.BatchNorm1d(48)                   # 3个分支各16通道 → 48
        )
        
        # 使用改进的 Bi-ResAtt-LSTM
        self.lstm = BiResAttLSTM(
            input_size=48,
            hidden_size=hidden_size,
            num_layers=2,
            dropout=0.2
        )

        # Transformer 编码器
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_size * 2,  # 匹配 BiLSTM 输出
                nhead=4,
                dim_feedforward=256,
                batch_first=True,
                activation=F.gelu
            ),
            num_layers=2
        )

        # 分类器
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear((input_dim // 4) * (hidden_size * 2), 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        

    def forward(self, x):
        x = self.tcn(x)                 # [B, 48, L]
        x = x.permute(0, 2, 1)          # [B, L, 48]

        lstm_out = self.lstm(x)         # [B, L, hidden_size*2]
        trans_out = self.transformer_encoder(lstm_out)  # [B, L, hidden_size*2]

        return self.classifier(trans_out)



In [3]:
from sklearn import metrics
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
# 设置超参数
batch_size = 64
epochs =15   #15 0.0005 85.43  15 0.0003 85.05  20 0.0005 85.20     25 0.0005 85.11     30 0.0008 84.69  10 0.0005 84.19 
learning_rate = 0.0005
k=10
# 交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=40)
criterion = nn.CrossEntropyLoss()
# 统计结果
oos_pred = []
# 初始化结果列表
oos_accuracies = []
last_fold_y_true = []
last_fold_y_pred = []

# 初始化模型
model = NSLKDDModel(input_dim=196, num_classes=10).to(device)
#criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)

# 遍历折叠
# 过采样少数类
#X_resampled, y_resampled = oversample.fit_resample(X, y_encoded)

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y_encoded), start=1):
#for fold, (train_idx, test_idx) in enumerate(kfold.split(X_resampled, y_resampled), start=1):
    # 划分数据集
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    #X_train, X_test = X_resampled[train_idx], X_resampled[test_idx]
    #y_train, y_test = y_resampled[train_idx], y_resampled[test_idx]

    # 过采样少数类
    X_train, y_train = oversample.fit_resample(X_train, y_train)

    # 转换为 PyTorch TensorDataset
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                torch.tensor(y_test, dtype=torch.long))

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Fold {fold}: {len(train_loader.dataset)} train samples, {len(test_loader.dataset)} validation samples")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        # 使用 tqdm 显示进度条
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch_data, batch_labels in progress_bar:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # 累积损失和准确性
            epoch_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

            # 更新进度条
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 计算每轮的平均损失和准确率
        epoch_loss /= len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")    

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)

            y_true.extend(batch_labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # 测试集各类别数量
    # test_class_counts = Counter(y_true)
    # print("\nTest Set Class Distribution:")
    # for label, count in sorted(test_class_counts.items()):
    #     print(f"  Class {label}: {count}")

    # 计算准确率
    acc = metrics.accuracy_score(y_true, y_pred)
    oos_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    # 保存最后一折的结果
    if fold == kfold.get_n_splits():
        last_fold_y_true = y_true
        last_fold_y_pred = y_pred

# 保存模型
model_save_path = "modelU12.pth"
torch.save(model, model_save_path)
print(f"Complete model saved to {model_save_path}")

# 输出每一折的准确率
print("Fold Accuracies:")
for i, acc in enumerate(oos_accuracies, start=1):
    print(f"  Fold {i}: {acc:.4f}")
 

Fold 1: 315449 train samples, 25768 validation samples


Epoch 1/15:   0%|          | 0/4929 [00:00<?, ?it/s]E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Convolution.cpp:1041.)
  return F.conv1d(input, weight, bias, self.stride,
E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
Epoch 1/15: 100%|██████████| 4929/4929 [00:41<00:00, 119.89it/s, loss=0.4649]


Epoch 1/15 - Loss: 0.2682, Accuracy: 0.8184


Epoch 2/15: 100%|██████████| 4929/4929 [00:41<00:00, 117.99it/s, loss=0.1927]


Epoch 2/15 - Loss: 0.2061, Accuracy: 0.8467


...

Epoch 13/15 - Loss: 0.1293, Accuracy: 0.8876


Epoch 14/15: 100%|██████████| 4929/4929 [00:40<00:00, 122.94it/s, loss=0.2059]


Epoch 14/15 - Loss: 0.1295, Accuracy: 0.8870


Epoch 15/15: 100%|██████████| 4929/4929 [00:39<00:00, 123.27it/s, loss=0.0519]


Epoch 15/15 - Loss: 0.1294, Accuracy: 0.8871
Fold 10 Accuracy: 0.8526
Complete model saved to modelU12.pth
Fold Accuracies:
  Fold 1: 0.8157
  Fold 2: 0.8228
  Fold 3: 0.8308
  Fold 4: 0.8287
  Fold 5: 0.8345
  Fold 6: 0.8385
  Fold 7: 0.8329
  Fold 8: 0.8423
  Fold 9: 0.8437
  Fold 10: 0.8526


In [4]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 分类类别
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms']

# 混淆矩阵（最后一折的结果）
last_cm = confusion_matrix(last_fold_y_true, last_fold_y_pred, labels=range(10))

print("\nLast Fold Confusion Matrix:")
print(last_cm)

report = classification_report(last_fold_y_true, last_fold_y_pred, target_names=categories)
print("\nClassification Report:")
print(report)

# 从混淆矩阵提取所有类别的统计信息
total_samples = last_cm.sum()  # 总样本数
correct_predictions = np.trace(last_cm)  # 对角线元素的和，即正确分类的总数

# 总体准确率（直接计算）
overall_accuracy = correct_predictions / total_samples

# 初始化总体指标
overall_TP = 0
overall_FN = 0
overall_FP = 0
overall_TN = 0


# 重新计算分类报告中的 TP、FP、FN、TN
detection_rates = {}
false_positive_rates = {}

for i, category in enumerate(categories):
    TP = last_cm[i, i]
    FN = last_cm[i, :].sum() - TP
    FP = last_cm[:, i].sum() - TP
    TN = total_samples - (TP + FP + FN)

    if category != "Normal":  # 统计攻击类型的总 TP 和 FN
        overall_TP += TP
        overall_FN += FN
    else:  # 统计正常类型的 TN 和 FP
        overall_TN += TN
        overall_FP += FP

    # 每类检测率和误报率
    detection_rates[category] = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    false_positive_rates[category] = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    print(f"Category: {category}")
    print(f"  Detection Rate (DR): {detection_rates[category]:.4f}")
    print(f"  False Positive Rate (FPR): {false_positive_rates[category]:.4f}")

# 总体检测率（攻击类型的 DR）和误报率（Normal 的 FPR）
overall_dr = overall_TP / (overall_TP + overall_FN) if (overall_TP + overall_FN) > 0 else 0.0
overall_fpr = overall_FP / (overall_FP + overall_TN) if (overall_FP + overall_TN) > 0 else 0.0

print("\nOverall Metrics:")
print(f"  Accuracy (Acc): {overall_accuracy:.4f}")
print(f"  Detection Rate (DR): {overall_dr:.4f}")
print(f"  False Positive Rate (FPR): {overall_fpr:.4f}")


Last Fold Confusion Matrix:
[[  41    0   12  166   35    0   14    0    0    0]
 [   0   32   17  145   38    0    0    0    0    0]
 [   0    1  251 1304   36    6   12   14   10    1]
 [   0    4  122 4056  103   19   66   59   15    8]
 [   0    0   19  195 1536    1  659    8    7    0]
 [   0    0    8   31    1 5839    3    4    1    0]
 [   2    0    5   23  273    1 8981   10    5    0]
 [   0    0   17  273    2    1    3 1103    0    0]
 [   0    0    3   10   13    0    9    5  111    0]
 [   0    0    0    0    0    0    0    0    0   18]]

Classification Report:
                precision    recall  f1-score   support

      Analysis       0.95      0.15      0.26       268
      Backdoor       0.86      0.14      0.24       232
           DoS       0.55      0.15      0.24      1635
      Exploits       0.65      0.91      0.76      4452
       Fuzzers       0.75      0.63      0.69      2425
       Generic       1.00      0.99      0.99      5887
        Normal       0.